# API Connection / Provider Navigate MCP — Design Spec

**Status:** draft for review  
**Date:** 2026-08-07  
**Repo surface:** `spur-notebook` Notebook MCP (`src/mcp/tools/api_connection.rs`)  
**Related:** `notebook_catalog` layered datasource navigation; skills `skill_navigate` / `skill_read`  
**Problem evidence:** `notebook_list_api_connections` ≈ **105 KB** (5 connections; ~52% `manifestToml`); `notebook_list_api_providers` ≈ **1.4 MB** (47 providers; **98%** full table schemas; stripe alone ≈ 840 KB)

## Profile pin (NS-Mermaid)

| Profile | Version | State | Use in this notebook |
|---|---:|---|---|
| `relational_lia` (`flowchart`) | 1 | **implemented** | Architecture, projection, schema gate, root parse, payload-budget policy |

Theory: Z3 `qf_lia_bool_int_enum`. Formal claims live **only** in native `ns_mermaid` cells (not Markdown mermaid fences).

External solve evidence for numeric constants: `solve_id = sol_8efee2c221604ad3` (payload budget 16_000; default limit 12; table-name sample ≤ 12–18).

## One-line decision

Replace dump-style `list_api_*` as the agent primary path with **search + one-hop navigate** tools that project **cards → index → table detail**, never embedding full column schemas or manifests on discovery hits.

## 1. Architecture decision — tool class

Agents discover providers/connections with **navigate** (search or hop).  
`list_*` remains only as a **summary** compatibility surface (no schemas).  
**Detail** reads (full columns, optional manifest) require an explicit root/table hop or status tool.

| Inputs | Tool class | Schema depth |
|---|---|---|
| no root, not dump | `navigate` (search) | none (card metadata + name samples) |
| root set, not dump | `navigate` (hop) | names-only index |
| dump intent, no root | `list_summary` | none |
| dump intent + root | `detail` | full columns |

Formal unit: `@spec API-NAV-ARCHITECTURE`.

In [ ]:
flowchart TD
    CTX["`@spec API-NAV-ARCHITECTURE
@type ToolClass = enum[navigate, list_summary, detail]
@type SchemaDepth = enum[none, names_only, full_columns]
@input has_root: Bool
@input wants_full_list_dump: Bool
@output tool: ToolClass
@output schema_depth: SchemaDepth`"]

    NAV_SEARCH["`@branch NAV_SEARCH
@when not has_root and not wants_full_list_dump
@ensures T_NAV: tool = navigate
@ensures D_NONE: schema_depth = none`"]

    NAV_HOP["`@branch NAV_HOP
@when has_root and not wants_full_list_dump
@ensures T_NAV2: tool = navigate
@ensures D_NAMES: schema_depth = names_only`"]

    LIST_SUM["`@branch LIST_SUMMARY
@when wants_full_list_dump and not has_root
@ensures T_LIST: tool = list_summary
@ensures D_NONE2: schema_depth = none`"]

    DETAIL["`@branch DETAIL_READ
@when wants_full_list_dump and has_root
@ensures T_DET: tool = detail
@ensures D_FULL: schema_depth = full_columns`"]

    CHECK["`@verify ARCH_DETERMINISTIC: prove determinism
@verify ARCH_COVERAGE: prove partition_coverage
@verify ARCH_EXCLUSIVE: prove partition_exclusive
@verify NAV_SEARCH_REACHABLE: witness branch NAV_SEARCH
@verify NAV_HOP_REACHABLE: witness branch NAV_HOP
@verify LIST_SUM_REACHABLE: witness branch LIST_SUMMARY
@verify DETAIL_REACHABLE: witness branch DETAIL_READ`"]

    CTX --> NAV_SEARCH --> CHECK
    CTX --> NAV_HOP --> CHECK
    CTX --> LIST_SUM --> CHECK
    CTX --> DETAIL --> CHECK

## 2. Goals and non-goals

### Goals

1. **Discoverability without dump** — agent can find a provider/connection by query in a few KB.
2. **Layered detail** — card → entity index → table schema, matching `notebook_catalog` / `skill_navigate`.
3. **Hard schema gate** — full `tables[].columns` (and `manifestToml`) never appear on search/list hits.
4. **Bounded payloads** — navigate search responses target ≤ **16 KB** agent-facing budget (solve model).
5. **Compat** — existing add/oauth/status/preview tools stay; list tools demote to summary or gain explicit `detail` opt-in.

### Non-goals

- Replacing `notebook_catalog` for *already-attached* datasources.
- BM25/vector index for provider catalog in v1 (substring match over name/displayName/category/table names is enough at ~47 providers).
- Changing Nango daemon RPC shape in v1 (project at MCP boundary).
- Auto-materializing OpenAPI bodies into the agent context.

### Measured baseline (2026-08-07 session)

| Tool | Bytes | Dominant field |
|---|---:|---|
| `notebook_list_api_connections` | 104_962 | `manifestToml` 54_779; tables/callables ~46k |
| `notebook_list_api_providers` | 1_392_697 | `tables` 1_364_356 |
| Projected meta+names (providers) | ~39_000 | — |
| Projected meta-only (providers) | ~18_000 | — |
| Projected slim connections | ~2_000–3_000 | — |

## 3. MCP tool contracts

### 3.1 `notebook_navigate_api_providers` (new, primary)

```
notebook_navigate_api_providers({
  query?: string,           // required when root omitted (non-empty)
  root?: string,            // "category:<cat>" | "<provider_key>" | "<provider_key>/<table>"
  limit?: int,              // default 12, min 1, max 12 (hard)
  include_lede?: bool,      // default true; lede ≤ 180 UTF-8 chars
  category?: string,
  tier?: "A" | "B",
  support_level?: string,
  auth_mode?: string,
  fulfillment_status?: string
})
```

| Mode | Input | Response shape |
|---|---|---|
| Search | `query` | `{ hits: ProviderCard[], total, truncated, next_queries }` |
| Category hop | `root: category:…` | provider cards in category |
| Entity hop | `root: <provider_key>` | `{ provider, tables: TableIndexEntry[], actions: ActionIndexEntry[] }` |
| Table hop | `root: <pk>/<table>` | full table schema (columns, path, method, filters) |

**ProviderCard** (never includes column schemas):  
`name`, `provider_key`, `display_name`, `category`, `tier`, `auth_mode`, `support_level`, `fulfillment_status`, `base_url?`, `table_count`, `action_count`, `table_names_sample[≤12]`, `action_names_sample[≤5]`, `credential_env_vars`, `lede?`

**TableIndexEntry:** `name`, `method`, `path` only.

### 3.2 `notebook_navigate_api_connections` (new, primary)

```
notebook_navigate_api_connections({
  query?: string,          // empty/omit = list all slim cards
  root?: string,            // "<connection_name>" | "<name>/<table>"
  limit?: int,              // default 12, max 12
  include_manifest?: bool   // default false; only on entity hop
})
```

| Mode | Response |
|---|---|
| Search/list | ConnectionCard: name, provider, group, status, missing_env_count, table_count, table_names, callable_table_functions |
| Entity hop | status + callables + table index; `manifestToml` only if `include_manifest` |
| Table hop | full table schema + invoke SQL |

### 3.3 Existing tools

| Tool | Change |
|---|---|
| `notebook_list_api_providers` | Summary projection by default (cards only); optional `detail: "full"` escape hatch **or** deprecate in descriptions |
| `notebook_list_api_connections` | Slim cards; no `manifestToml` by default |
| `notebook_api_connection_status` | Keep; ensure it does not force full template dump unless needed |
| add / oauth / preview | Unchanged |

### 3.4 Matching (v1)

Case-insensitive substring over `name | displayName | category | providerKey | table names | action names`. Rank: exact name > prefix > substring; stable secondary sort by name.

## 4. Projection partition (card / index / table detail)

Every navigate response selects exactly one projection tier.  
`include_manifest` may only apply on entity hop (index), never on cards or table leaves.

Formal unit: `@spec API-NAV-PROJECTION`.

In [ ]:
flowchart TD
    CTX["`@spec API-NAV-PROJECTION
@type Projection = enum[card, index, table_detail]
@type EmitColumns = enum[omit, include]
@type EmitManifest = enum[omit, include]
@input is_table_leaf: Bool
@input is_entity_hop: Bool
@input include_manifest_flag: Bool
@output projection: Projection
@output emit_columns: EmitColumns
@output emit_manifest: EmitManifest`"]

    CARD["`@branch CARD
@when not is_table_leaf and not is_entity_hop
@ensures P_CARD: projection = card
@ensures C_OMIT: emit_columns = omit
@ensures M_OMIT: emit_manifest = omit`"]

    INDEX_PLAIN["`@branch INDEX_PLAIN
@when is_entity_hop and not is_table_leaf and not include_manifest_flag
@ensures P_IDX: projection = index
@ensures C_OMIT2: emit_columns = omit
@ensures M_OMIT2: emit_manifest = omit`"]

    INDEX_MANIFEST["`@branch INDEX_MANIFEST
@when is_entity_hop and not is_table_leaf and include_manifest_flag
@ensures P_IDX2: projection = index
@ensures C_OMIT3: emit_columns = omit
@ensures M_INC: emit_manifest = include`"]

    TABLE["`@branch TABLE_DETAIL
@when is_table_leaf
@ensures P_TAB: projection = table_detail
@ensures C_INC: emit_columns = include
@ensures M_OMIT3: emit_manifest = omit`"]

    CHECK["`@verify PROJ_DETERMINISTIC: prove determinism
@verify PROJ_COVERAGE: prove partition_coverage
@verify PROJ_EXCLUSIVE: prove partition_exclusive
@verify CARD_REACHABLE: witness branch CARD
@verify INDEX_PLAIN_REACHABLE: witness branch INDEX_PLAIN
@verify INDEX_MANIFEST_REACHABLE: witness branch INDEX_MANIFEST
@verify TABLE_REACHABLE: witness branch TABLE_DETAIL`"]

    CTX --> CARD --> CHECK
    CTX --> INDEX_PLAIN --> CHECK
    CTX --> INDEX_MANIFEST --> CHECK
    CTX --> TABLE --> CHECK

## 5. Schema emission gate (hard invariant)

Full column arrays are **forbidden** on card and index projections; **allowed** only on table_detail.

This is the load-bearing anti-noise rule that kills the 1.4 MB list path.

Formal unit: `@spec API-NAV-SCHEMA-GATE`.

In [ ]:
flowchart TD
    CTX["`@spec API-NAV-SCHEMA-GATE
@type Projection = enum[card, index, table_detail]
@type ColumnEmit = enum[forbidden, allowed]
@input projection: Projection
@output column_emit: ColumnEmit`"]

    FORBID["`@branch FORBID_COLUMNS
@when projection = card or projection = index
@ensures FORBID: column_emit = forbidden`"]

    ALLOW["`@branch ALLOW_COLUMNS
@when projection = table_detail
@ensures ALLOW: column_emit = allowed`"]

    CHECK["`@verify GATE_DETERMINISTIC: prove determinism
@verify GATE_COVERAGE: prove partition_coverage
@verify GATE_EXCLUSIVE: prove partition_exclusive
@verify FORBID_REACHABLE: witness branch FORBID_COLUMNS
@verify ALLOW_REACHABLE: witness branch ALLOW_COLUMNS`"]

    CTX --> FORBID --> CHECK
    CTX --> ALLOW --> CHECK

## 6. Root parse (navigate hop grammar)

`root` is optional. When present:

| Form | Kind |
|---|---|
| (absent) | search / list cards |
| `category:<name>` | category hop |
| `<entity>` (no `/`) | entity hop (provider_key or connection name) |
| `<entity>/<table>` | table leaf |

Formal unit: `@spec API-NAV-ROOT-PARSE`.

In [ ]:
flowchart TD
    CTX["`@spec API-NAV-ROOT-PARSE
@type RootKind = enum[none, category, entity, table_leaf]
@input has_root_string: Bool
@input has_slash: Bool
@input is_category_prefix: Bool
@output root_kind: RootKind`"]

    NONE["`@branch ROOT_NONE
@when not has_root_string
@ensures K_NONE: root_kind = none`"]

    CAT["`@branch ROOT_CATEGORY
@when has_root_string and not has_slash and is_category_prefix
@ensures K_CAT: root_kind = category`"]

    ENT["`@branch ROOT_ENTITY
@when has_root_string and not has_slash and not is_category_prefix
@ensures K_ENT: root_kind = entity`"]

    LEAF["`@branch ROOT_TABLE
@when has_root_string and has_slash
@ensures K_LEAF: root_kind = table_leaf`"]

    CHECK["`@verify ROOT_DETERMINISTIC: prove determinism
@verify ROOT_COVERAGE: prove partition_coverage
@verify ROOT_EXCLUSIVE: prove partition_exclusive
@verify NONE_REACHABLE: witness branch ROOT_NONE
@verify CAT_REACHABLE: witness branch ROOT_CATEGORY
@verify ENT_REACHABLE: witness branch ROOT_ENTITY
@verify LEAF_REACHABLE: witness branch ROOT_TABLE`"]

    CTX --> NONE --> CHECK
    CTX --> CAT --> CHECK
    CTX --> ENT --> CHECK
    CTX --> LEAF --> CHECK

## 7. Payload budget policy

Numeric constants from independent `solve_constraints` (`solve_id = sol_8efee2c221604ad3`):

| Constant | Value | Role |
|---|---:|---|
| `payload_budget` | 16_000 | Soft agent-facing budget for search hits |
| `limit` default/max | 12 | Hits per navigate search |
| `max_table_names_sample` | 12 | Names on a card |
| `lede_chars` | ≤ 180 | Optional lede |
| card overhead model | ~400 + lede + 20×sample | Feasibility sketch |

Empirical: 12 sample cards ≪ 16 KB; full 47 sample cards ~20–40 KB; **schema-free** is what matters.

The formal cell below proves the **policy partition** (fits vs exceeds) under the three boolean gates the implementation must enforce — not the nonlinear byte formula (QF_LIA forbids free var-var mul in ns-mermaid).

Formal unit: `@spec API-NAV-PAYLOAD-BUDGET`.

In [ ]:
flowchart TD
    CTX["`@spec API-NAV-PAYLOAD-BUDGET
@type Fit = enum[fits, exceeds]
@input limit_le_12: Bool
@input sample_le_12: Bool
@input no_full_schemas_on_hits: Bool
@output status: Fit`"]

    FITS["`@branch FITS
@when limit_le_12 and sample_le_12 and no_full_schemas_on_hits
@ensures OK: status = fits`"]

    EXCEEDS["`@branch EXCEEDS
@when not (limit_le_12 and sample_le_12 and no_full_schemas_on_hits)
@ensures BAD: status = exceeds`"]

    CHECK["`@verify BUDGET_DETERMINISTIC: prove determinism
@verify BUDGET_COVERAGE: prove partition_coverage
@verify BUDGET_EXCLUSIVE: prove partition_exclusive
@verify FITS_REACHABLE: witness branch FITS
@verify EXCEEDS_REACHABLE: witness branch EXCEEDS
@verify EACH_STATUS: witness each status`"]

    CTX --> FITS --> CHECK
    CTX --> EXCEEDS --> CHECK

## 8. Implementation seams (spur-notebook)

| Area | Path / symbol |
|---|---|
| Tool schemas + handlers | `spur-notebook/src/mcp/tools/api_connection.rs` |
| Tool registration | `src/mcp/tools/mod.rs`, `src/mcp/server.rs` |
| Connection enrichment (today dumps template) | `enriched_connection` — **must** gain slim projector |
| Provider source | daemon `ListNangoProviders` — project at MCP boundary in v1 |
| Precedent hop API | `src/mcp/tools/notebook_catalog.rs` |

### Suggested phases

1. **Projectors** — `ProviderCard`, `ConnectionCard`, `TableIndexEntry`; unit tests assert no `columns` / `manifestToml` keys on cards.
2. **navigate_api_providers** + **navigate_api_connections** — register tools; root parser; substring ranker.
3. **Demote list_*** — summary default; description text points agents to navigate.
4. **Tests** — fixture with multi-table provider (stripe-like) proves list/search payload size floor; table hop returns columns.
5. **Agent skill copy** — notebook data-app / skills that mention list_api_* should prefer navigate.

### Acceptance tests (implementation)

- `navigate_api_providers({query:"stripe"})` returns ≤ limit cards, **zero** nested `columns` arrays.
- `navigate_api_providers({root:"stripe"})` returns table index; still no columns.
- `navigate_api_providers({root:"stripe/invoices"})` (or real table name) returns columns.
- `navigate_api_connections({})` for N saved connections returns no `manifestToml`.
- `list_api_providers` without detail flag is schema-free (if demoted).
- Golden size: provider search payload under 16 KB for limit=12 on current catalog.

## 9. Agent workflow (post-change)

```
1. notebook_navigate_api_providers({ query: "billing invoices stripe" })
   → cards with table_count + name samples

2. notebook_navigate_api_providers({ root: "stripe" })
   → table/action index (name, method, path)

3. notebook_navigate_api_providers({ root: "stripe/<table>" })
   → full columns + filters

4. notebook_add_api_connection / oauth / status as today
```

Saved connections:

```
navigate_api_connections()                     // slim cards
navigate_api_connections({ root: "github" })   // callables, no manifest
api_connection_status({ name: "github" })      // status envelope
```

## 10. Risks

| Risk | Mitigation |
|---|---|
| Agents keep calling legacy list dump | Demote payload + update tool descriptions; skill guidance |
| Stripe-scale entity hop still large | Index is name/method/path only; still truncate with next_queries if needed |
| Substring false positives | Rank + limit 12; optional filters |
| Dual source of truth (list vs navigate) | List becomes summary of same projectors |
| Daemon full payload still expensive | v1 OK; v2 slim daemon RPC if profiling shows need |

## 11. Executable NS-Mermaid evidence

All formal cells preflighted with `notebook_ns_mermaid_check`, then executed with `notebook_run_cell`. **All five verified=true with proof_fresh=true** after sequential re-run.

Profile pin: `relational_lia` v1 · registry_schema_version 1 · Z3 `qf_lia_bool_int_enum`

| @spec | Cell id | Obligations matched | source_hash (sha256) | ir_hash | report_hash |
|---|---|---:|---|---|---|
| `API-NAV-ARCHITECTURE` | `7a6502f1-9940-4a7e-8cf7-672b2ae965e2` | 7/7 | `39e95a5cda1fa1bcd92bae95e602a2a0f29e765f3727bf2700e828c4b3a93107` | `d349eda3923bdb8098c9758cd7f8163ed760d38e2c3b8f249c65f631ec219fb1` | `3abc7ae9651f3110ca2aba134b48d6788c84576d37df64485ca15489c0ede63c` |
| `API-NAV-PROJECTION` | `5a8bfb4c-d484-4b1c-8779-c4573a5db66b` | 7/7 | `06989856d820be426fe2c6b24ae7cd27e3975dfa5e7797bcb5a4bd5eb679ea42` | `4035c92b614e3fd58aba1fc8b6e9b7c4f82cc6c3d191b6bb35f1c56cf35b2ea5` | `289470c71fd6c0faca7562ea61deb5befb639cc4affe18fe7fdbf58773213e73` |
| `API-NAV-SCHEMA-GATE` | `b509c2ff-df27-4077-ac6c-c4f4bc76a8a9` | 5/5 | `cd02a07cf5f0c6fb2f747b377cd05a61dc7fb001f8d9032370d6740811a4f03c` | `9da47e5f6b75d4d9b026e1ce03d05ccbb163e8c1502ae5919a6a4621673806f5` | `03d183fc91796d91c2ca1909cd81dda099e0e823d1023ee777f5238c7dc3d8d0` |
| `API-NAV-ROOT-PARSE` | `685f558f-e9b3-4f98-9cd0-efa9da6f6b31` | 7/7 | `cfe0aa6c12a6d9fefef0f77e491fd9b93ee26d6a05b14c59f5a0eecbc2df4ca7` | `c38d81c9a1912623a2b387f1166402be00c838d8f4222cd3711cce679bdc5ff2` | `043e66b5385ee0a6d3b603c0b2c29dd3ae141f9919b7b58f3f938d3adf274334` |
| `API-NAV-PAYLOAD-BUDGET` | `2268215e-3f7b-464d-800b-9346d9f8ca01` | 7/7 | `7e81e3985a64dba96bc5cee2906239411c65f1e54668b01f8a60a723948ad07e` | `822d73f246daccfb1d28971378cb84e257c536994eaa37ef70ab6f4c9ca4c8cb` | `a652be5d0e080aa7b4ccbfbf0895b88bfe295f9e43c07562fee4a7acfb15f151` |

External numeric solve: `solve_id = sol_8efee2c221604ad3` → `{payload_budget:16000, limit:12, max_table_names:18, lede_chars:181}`.

**Immutable contract surface for implementation:** the five `@spec` IDs above. Workers must not re-introduce full schema dumps on card/index projections.